In [ ]:
import sys
import torch

print("Python:", sys.version)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

In [ ]:
%pip install ipywidgets

In [ ]:
%pip install "numpy==2.2.6"

In [ ]:
%pip install "matchms==0.30.2"

In [ ]:
%pip install "ms2deepscore==2.5.5" pandas pyarrow numpy matplotlib 

In [ ]:
from pathlib import Path
import glob, json, re, time, gc, hashlib, platform
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
import glob

# Experiment settings
MODE = '5pct'  # Change to '5pct' or '100pct' later

# Data files are in the same folder as this notebook
DATA_DIR = Path('.')

# Full dataset (only needed for the 100% experiment)
FULL_GLOB = r'model_datasets/spectra_train/*.parquet'

# Pretrained MS2DeepScore model
MODEL_FILE = Path('ms2deepscore_model.pt')

# Where results will be saved
OUTPUT_DIR = Path('ms2deepscore_results')
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# Processing settings
EMBED_BATCH = 256
QUERY_BATCH = 16
SPECTRUM_TOPK = 512

# Smoke test settings
MAX_SMOKE_TRAIN = 500
MAX_SMOKE_VALID = 100

# Check experiment mode
assert MODE in ('smoke', '5pct', '100pct')

# Check dataset files
assert (DATA_DIR / 'train.parquet').is_file(), \
    f'Cannot find {DATA_DIR}/train.parquet'

assert (DATA_DIR / 'validation.parquet').is_file(), \
    f'Cannot find {DATA_DIR}/validation.parquet'

# Check pretrained model
assert MODEL_FILE.is_file(), \
    f'Download the compatible pretrained checkpoint: {MODEL_FILE}'

# Locate full training files
full_files = sorted(glob.glob(FULL_GLOB))

if MODE == '100pct':
    assert full_files, \
        f'No 100% Parquet partitions match: {FULL_GLOB}'

    excluded = {
        (DATA_DIR / 'train.parquet').resolve(),
        (DATA_DIR / 'validation.parquet').resolve()
    }

    assert not any(Path(p).resolve() in excluded for p in full_files), \
        'FULL_GLOB points at the 5% files!'

print('Mode:', MODE)
print('Training data found:', (DATA_DIR / 'train.parquet').is_file())
print('Validation data found:', (DATA_DIR / 'validation.parquet').is_file())
print('100% partitions located:', len(full_files))
print('Outputs:', OUTPUT_DIR)


In [ ]:
train5 = pd.read_parquet(DATA_DIR / 'train.parquet')
validation_all = pd.read_parquet(DATA_DIR / 'validation.parquet')
required_cols = {'inchikey14', 'mzs', 'intensities', 'precursor_mz', 'ionization_mode', 'source_part', 'row_id'}
for name, df in [('train', train5), ('validation', validation_all)]:
    missing = required_cols - set(df.columns)
    if missing: raise ValueError(f'{name} missing columns: {missing}')

# Molecule identity is InChIKey14, same as your classmates' classifiers.
known = set(train5['inchikey14'].dropna().astype(str))
valid = validation_all.loc[validation_all['inchikey14'].astype(str).isin(known)].copy()
print('5% reference spectra:', len(train5))
print('Original validation spectra:', len(validation_all))
print('Fixed known-molecule validation spectra:', len(valid))
print('Known-molecule validation coverage:', round(len(valid) / len(validation_all)*100, 2), '%')
assert len(valid) > 0 and len(train5) > 0

def norm_part(x):
    s = str(x).strip().lower()
    if s.isdigit(): return f'{int(s):03d}'
    m = re.search(r'part[_ -]?(\d+)', s)
    return f'{int(m.group(1)):03d}' if m else s

def row_record_key(src, row):
    if pd.isna(src) or pd.isna(row): raise ValueError('Missing source_part/row_id; cannot prove non-leakage')
    return (norm_part(src), str(int(row)) if isinstance(row, (int, float, np.integer, np.floating)) and float(row).is_integer() else str(row))

val_ids = set(row_record_key(a,b) for a,b in zip(validation_all.source_part, validation_all.row_id))
train5_ids = set(row_record_key(a,b) for a,b in zip(train5.source_part,train5.row_id))
assert not (val_ids & train5_ids), 'Training and validation share records!'
print('Held-out record IDs for exclusion from 100%:', len(val_ids))
print('5% training / validation record intersection:', len(train5_ids & val_ids))

In [ ]:
from matchms import Spectrum
from ms2deepscore import MS2DeepScore
from ms2deepscore.models import load_model

def interpret_ionmode(mode, charge):
    s = str(mode).strip().lower()
    if s in ('positive','pos','+','esi+','positive ion mode') or ('+' in s and '-' not in s): return 'positive'
    if s in ('negative','neg','-','esi-','negative ion mode') or ('-' in s and '+' not in s): return 'negative'
    try:
        c = float(charge)
        if c > 0: return 'positive'
        if c < 0: return 'negative'
    except (TypeError, ValueError): pass
    return None

def make_spectrum(row):
    mz = np.asarray(row.mzs, dtype=np.float64).reshape(-1)
    intens = np.asarray(row.intensities, dtype=np.float64).reshape(-1)
    if len(mz) != len(intens) or len(mz)==0: return None
    mask = np.isfinite(mz) & np.isfinite(intens) & (mz > 0) & (intens > 0)
    mz, intens = mz[mask], intens[mask]
    if len(mz) == 0: return None
    order = np.argsort(mz)
    mz, intens = mz[order], intens[order]
    intens = intens / intens.max()
    precursor = float(row.precursor_mz) if pd.notna(row.precursor_mz) else None
    ionmode = interpret_ionmode(row.ionization_mode, getattr(row, 'charge', None))
    if precursor is None or not np.isfinite(precursor) or ionmode is None: return None
    return Spectrum(mz=mz, intensities=intens,
                    metadata={'precursor_mz':precursor,'ionmode':ionmode},
                    metadata_harmonization=True)

print('Example:', make_spectrum(next(train5.head(1).itertuples(index=False))))
model = load_model(str(MODEL_FILE))
# The package version and checkpoint are pinned to the version documented on Zenodo.
if torch.cuda.is_available() and hasattr(model, 'to'):
    model = model.to('cuda')
if hasattr(model, 'eval'): model.eval()
ms2ds = MS2DeepScore(model)
print('Pretrained model loaded. No network weights will be trained here.')

In [ ]:
COLUMNS = ['inchikey14','mzs','intensities','precursor_mz','ionization_mode','charge','source_part','row_id']

def file_batches(paths, max_rows=None):
    seen = 0
    for path in paths:
        pf = pq.ParquetFile(str(path))
        available = set(pf.schema_arrow.names)
        required = set(COLUMNS) - {'charge'}
        if not required.issubset(available):
            raise ValueError(f'{path} missing required fields: {required-available}. Cannot safely exclude validation records.')
        cols = [a for a in COLUMNS if a in available]
        for rb in pf.iter_batches(batch_size=EMBED_BATCH, columns=cols):
            frame = rb.to_pandas()
            if max_rows is not None and seen + len(frame) > max_rows:
                frame = frame.iloc[:max_rows-seen]
            if len(frame): yield frame
            seen += len(frame)
            if max_rows is not None and seen >= max_rows: return

def write_cache(paths, tag, exclude_validation, limit=None):
    emb_file = OUTPUT_DIR / f'{tag}_embeddings.npy'
    labels_file = OUTPUT_DIR / f'{tag}_labels.npy'
    meta_file = OUTPUT_DIR / f'{tag}_info.json'
    if all(p.exists() for p in (emb_file,labels_file,meta_file)):
        info = json.loads(meta_file.read_text())
        assert info['checkpoint_path'] == str(MODEL_FILE.resolve()), 'Cached embeddings from a different pretrained model path'
        assert info['source_files'] == [str(Path(x).resolve()) for x in paths], 'Cached embeddings from different source files'
        assert info['max_rows'] == limit and info['excluded_validation'] == exclude_validation, 'Cache settings mismatch'
        print('Using existing embeddings:', tag, info['count'])
        return emb_file, labels_file, info

    max_slots = sum(pq.ParquetFile(str(p)).metadata.num_rows for p in paths)
    if limit is not None: max_slots = min(max_slots, limit)
    embs = labels = None
    count = read = skipped = excluded = 0
    checked_full_train_ids = set()
    checked_full_validation_ids = set()
    start = time.perf_counter()
    for frame in file_batches(paths, limit):
        read += len(frame)
        if exclude_validation:
            keys = [row_record_key(a,b) for a,b in zip(frame.source_part,frame.row_id)]
            checked_full_train_ids.update(x for x in keys if x in train5_ids)
            checked_full_validation_ids.update(x for x in keys if x in val_ids)
            keep = [x not in val_ids for x in keys]
            excluded += len(keep) - sum(keep)
            frame = frame.loc[keep]
        if len(frame) == 0: continue
        sp, labs = [], []
        for r in frame.itertuples(index=False):
            s = make_spectrum(r)
            mol = str(r.inchikey14)
            if s is None or not mol or mol == 'nan':
                skipped += 1
                continue
            sp.append(s); labs.append(mol)
        if not sp: continue
        block = np.asarray(ms2ds.get_embedding_array(sp), dtype=np.float32)
        if block.ndim != 2 or block.shape[0] != len(sp) or not np.isfinite(block).all():
            raise ValueError('MS2DeepScore returned invalid embeddings. Examine input data and model compatibility.')
        if embs is None:
            embs = np.lib.format.open_memmap(emb_file,mode='w+',dtype=np.float32,shape=(max_slots,block.shape[1]))
            labels = np.lib.format.open_memmap(labels_file,mode='w+',dtype='S14',shape=(max_slots,))
        embs[count:count+len(sp)] = block
        labels[count:count+len(sp)] = np.asarray(labs,dtype='S14')
        count += len(sp)
        if read % 100000 < EMBED_BATCH: print(f' {tag}: read {read:,}, embedded {count:,}, skipped {skipped:,}, excluded {excluded:,}')
    if embs is None: raise ValueError('No valid spectra could be embedded')
    # Hard stop if identifiers cannot connect the full source to the 5% samples.
    # This protects against silently retaining held-out records with changed IDs.
    if exclude_validation:
        if checked_full_train_ids != train5_ids or checked_full_validation_ids != val_ids:
            raise ValueError(
                '100% source partition IDs do not match the original 5% split. '
                f'Found {len(checked_full_train_ids)}/{len(train5_ids)} train IDs and '
                f'{len(checked_full_validation_ids)}/{len(val_ids)} validation IDs. '
                'Do not evaluate until you reconcile source_part and row_id between files.'
            )
    embs.flush(); labels.flush()
    info = {'count':count,'read':read,'skipped_invalid':skipped,'excluded_validation':exclude_validation,
            'excluded_count':excluded,'dimension':int(embs.shape[1]),'embedding_seconds':time.perf_counter()-start,
            'checkpoint_path':str(MODEL_FILE.resolve()),'source_files':[str(Path(x).resolve()) for x in paths],
            'max_rows':limit}
    meta_file.write_text(json.dumps(info,indent=2))
    del embs,labels
    return emb_file,labels_file,info

reference_paths = [DATA_DIR/'train.parquet'] if MODE != '100pct' else list(map(Path,full_files))
# IMPORTANT: check identity mapping BEFORE expensive 100% embedding.
if MODE == '100pct':
    found_train, found_val = set(), set()
    for path in reference_paths:
        pf = pq.ParquetFile(str(path))
        needed = {'source_part', 'row_id'}
        if not needed.issubset(pf.schema_arrow.names):
            raise ValueError(f'{path} lacks {needed-set(pf.schema_arrow.names)}. Cannot prove non-leakage: ask the data preparer for matching record IDs.')
        for rb in pf.iter_batches(batch_size=100_000, columns=['source_part', 'row_id']):
            ids = [row_record_key(a,b) for a,b in zip(rb.column(0).to_pylist(),rb.column(1).to_pylist())]
            found_train.update(x for x in ids if x in train5_ids)
            found_val.update(x for x in ids if x in val_ids)
    print(f'Full-dataset ID preflight: 5% train {len(found_train)}/{len(train5_ids)}, '
          f'held-out validation {len(found_val)}/{len(val_ids)}')
    if found_train != train5_ids or found_val != val_ids:
        raise ValueError('STOP: the source partitions and 5% data do not have matching source_part/row_id keys. Reconcile original record IDs before running 100%.')
smoke_limit = MAX_SMOKE_TRAIN if MODE=='smoke' else None
ref_tag = MODE
ref_e_file, ref_l_file, ref_info = write_cache(reference_paths, ref_tag, exclude_validation=(MODE=='100pct'),limit=smoke_limit)
print('Reference:', ref_info)

In [ ]:
val_paths = [DATA_DIR/'validation.parquet']
# Use the already-filtered validation dataframe, NOT the full 22,963 records.
if MODE=='smoke': valid_work = valid.head(MAX_SMOKE_VALID).copy()
else: valid_work = valid.copy()

sp, val_true, val_ids_kept = [], [], []
for r in valid_work.itertuples(index=False):
    spectrum = make_spectrum(r)
    if spectrum is None: continue
    sp.append(spectrum)
    val_true.append(str(r.inchikey14))
    val_ids_kept.append(row_record_key(r.source_part,r.row_id))
if not sp: raise RuntimeError('No valid held-out spectra passed MS2DeepScore preprocessing')
start = time.perf_counter()
val_emb_chunks = []
for i in range(0,len(sp),EMBED_BATCH):
    val_emb_chunks.append(np.asarray(ms2ds.get_embedding_array(sp[i:i+EMBED_BATCH]),dtype=np.float32))
val_emb = np.concatenate(val_emb_chunks)
val_embedding_time = time.perf_counter() - start
print('Validation embeddings:',val_emb.shape,'| embedding seconds:',round(val_embedding_time,2))
np.save(OUTPUT_DIR/f'{MODE}_validation_embeddings.npy',val_emb)

In [ ]:
ref_count = ref_info['count']
reference = np.load(ref_e_file, mmap_mode='r')[:ref_count]
ref_labels = np.load(ref_l_file, mmap_mode='r')[:ref_count]
assert reference.shape[1] == val_emb.shape[1]

search_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
print('Similarity-search device:', search_device)

# Stage all embeddings on GPU if possible; on CPU this may need several GB RAM.
# This is separate from the MS2DeepScore embedding-generation step.
ref_tensor = F.normalize(torch.from_numpy(np.asarray(reference)).to(search_device),dim=1)
true_lab = np.asarray(val_true)
predictions = []
unique_counts = []
search_start = time.perf_counter()

for start_idx in range(0,len(val_emb),QUERY_BATCH):
    q = torch.as_tensor(val_emb[start_idx:start_idx+QUERY_BATCH],dtype=torch.float32,device=search_device)
    q = F.normalize(q,dim=1)
    with torch.no_grad():
        scores = q @ ref_tensor.T
        k = min(SPECTRUM_TOPK, ref_count)
        top_ids = torch.topk(scores,k=k,dim=1).indices.cpu().numpy()
    for row in top_ids:
        # Highest similarity first; remove duplicate molecule identities.
        ranked = []
        seen = set()
        for idx in row:
            mol = ref_labels[int(idx)].decode('utf-8')
            if mol not in seen:
                seen.add(mol)
                ranked.append(mol)
            if len(ranked)>=25: break
        unique_counts.append(len(ranked))
        predictions.append(ranked)
    del q,scores,top_ids
if search_device.type=='cuda': torch.cuda.synchronize()
search_seconds = time.perf_counter()-search_start
peak_gpu_gib = torch.cuda.max_memory_allocated()/1024**3 if search_device.type=='cuda' else None
print('Search seconds:',round(search_seconds,2),'| PyTorch peak CUDA allocation (GiB):',peak_gpu_gib)
print('Results with 25 distinct candidate molecules:',sum(n>=25 for n in unique_counts),'/',len(unique_counts))

In [ ]:
ref_molecules = set(x.decode('utf-8') for x in np.unique(ref_labels))
coverage = float(np.mean([v in ref_molecules for v in true_lab]))

def topk_rate(k):
    eligible = [i for i,x in enumerate(predictions) if len(x)>=k]
    if not eligible: return None,len(eligible)
    return float(np.mean([true_lab[i] in predictions[i][:k] for i in eligible])),len(eligible)

accuracy = {f'top{k}':topk_rate(k)[0] for k in (1,5,25)}
metrics = {'mode':MODE,'reference_spectra':ref_count,'reference_molecules':len(ref_molecules),
           'validation_spectra_evaluated':len(true_lab),'reference_coverage':coverage,
           'embedding_seconds_reference':ref_info['embedding_seconds'],
           'embedding_seconds_validation':val_embedding_time,
           'search_seconds':search_seconds,'peak_pytorch_cuda_memory_gib':peak_gpu_gib,
           'checkpoint':str(MODEL_FILE),'model_retrained':False,
           **accuracy,
           **{f'eligible_top{k}':topk_rate(k)[1] for k in (1,5,25)}}
print(json.dumps(metrics,indent=2))
(OUTPUT_DIR/f'{MODE}_metrics.json').write_text(json.dumps(metrics,indent=2))
pd.DataFrame({'target_inchikey14':true_lab,
              'top1':[(p[0] if p else '') for p in predictions],
              'predicted_top25': [';'.join(p) for p in predictions],
              'n_unique_candidates':unique_counts}).to_csv(OUTPUT_DIR/f'{MODE}_predictions.csv',index=False)

# Save one row per run for group-level comparison (append without duplicate mode).
summary_csv = OUTPUT_DIR/'comparison.csv'
new = pd.DataFrame([metrics])
if summary_csv.is_file():
    old = pd.read_csv(summary_csv)
    old = old.loc[old['mode']!=MODE]
    new = pd.concat([old,new],ignore_index=True)
new.to_csv(summary_csv,index=False)
print('Saved:',summary_csv)

In [ ]:
summary_csv = OUTPUT_DIR/'comparison.csv'
if summary_csv.is_file():
    results = pd.read_csv(summary_csv)
    real = results[results['mode'].isin(['5pct','100pct'])].copy()
    display(real[['mode','reference_spectra','reference_molecules','validation_spectra_evaluated','reference_coverage','top1','top5','top25','search_seconds']])
    if len(real):
        ax=(real.set_index('mode')[['top1','top5','top25']]*100).plot(kind='bar',rot=0)
        ax.set_ylabel('Identification accuracy (%)')
        ax.set_title('Same validation spectra; pretrained MS2DeepScore; varying reference size')
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR/'accuracy_comparison.png',dpi=180)
        plt.show()

In [ ]:
import pandas as pd

results = pd.read_csv("ms2deepscore_results/comparison.csv")

print("Number of result rows:", len(results))
print(results.to_string(index=False))